# Managing short-term memory: summarising

When the chat gets long, the LLM writes a **summary** of the older messages, then those messages are
deleted. The summary is kept in the state and given to the LLM on every turn, so old facts survive
while the prompt stays small. It's trimming + deleting, without forgetting.

```
START -> chat_node --(more than 6 messages?)--> summarize -> END
                   --(no)---------------------------------> END
```

In [1]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage, RemoveMessage
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

model = ChatGroq(model='openai/gpt-oss-20b', reasoning_effort='low')

In [2]:
# MessagesState + one extra key for the running summary
class ChatState(MessagesState):
    summary: str


MAX_MESSAGES = 6  # summarise once the chat has more than this
KEEP_LAST = 2     # messages kept word for word after summarising


def chat_node(state: ChatState):
    messages = state['messages']
    if state.get('summary'):
        # the summary goes in front, so the LLM 'remembers' what was deleted
        messages = [SystemMessage(f"Summary of the earlier conversation: {state['summary']}")] + messages
    return {'messages': [model.invoke(messages)]}


def summarize(state: ChatState):
    old_summary = state.get('summary', '')
    if old_summary:
        prompt = f'This is the summary so far: {old_summary}\nExtend it with the new messages above. Keep names and facts about the user.'
    else:
        prompt = 'Summarise the conversation above in a few lines. Keep names and facts about the user.'
    new_summary = model.invoke(state['messages'] + [HumanMessage(prompt)]).content

    # delete everything except the last KEEP_LAST messages; their content now lives in the summary
    to_delete = [RemoveMessage(id=m.id) for m in state['messages'][:-KEEP_LAST]]
    return {'summary': new_summary, 'messages': to_delete}


def should_summarize(state: ChatState):
    return 'summarize' if len(state['messages']) > MAX_MESSAGES else END

In [3]:
graph = StateGraph(ChatState)
graph.add_node('chat_node', chat_node)
graph.add_node('summarize', summarize)
graph.add_edge(START, 'chat_node')
graph.add_conditional_edges('chat_node', should_summarize, ['summarize', END])
graph.add_edge('summarize', END)

chatbot = graph.compile(checkpointer=InMemorySaver())
config = {'configurable': {'thread_id': 'summary-1'}}

In [4]:
def chat(text):
    result = chatbot.invoke({'messages': [HumanMessage(content=text)]}, config=config)
    print(f'You: {text}\nAI:  {result["messages"][-1].content[:110]}')
    print(f'     (stored {len(result["messages"])} messages | summary: {"yes" if result.get("summary") else "no"})\n')


chat('Hi! My name is Priyansh and I live in Jaipur.')
chat('I am learning LangGraph.')
chat('Give me one short fact about Paris.')
chat('Give me one short fact about Tokyo.')  # 8 messages > 6 -> summarise, keep the last 2
chat('Give me one short fact about Cairo.')

You: Hi! My name is Priyansh and I live in Jaipur.
AI:  Hey Priyansh! 👋 Nice to meet you. How’s Jaipur treating you today?
     (stored 2 messages | summary: no)



You: I am learning LangGraph.
AI:  That’s awesome! LangGraph is a powerful framework for building conversational AI and knowledge‑graph‑driven ap
     (stored 4 messages | summary: no)



You: Give me one short fact about Paris.
AI:  Paris is home to the world’s tallest building, the Eiffel Tower, which reaches 324 m (1,063 ft) above ground.
     (stored 6 messages | summary: no)



You: Give me one short fact about Tokyo.
AI:  Tokyo is the world’s most populous metropolitan area, with over 37 million residents in the Greater Tokyo Area
     (stored 2 messages | summary: yes)



You: Give me one short fact about Cairo.
AI:  Cairo is the largest city in Africa, with a population of around 20 million residents.
     (stored 4 messages | summary: yes)



In [5]:
# The intro messages are deleted, but the facts survive in the summary
print('SUMMARY:\n', chatbot.get_state(config).values['summary'], '\n')
chat('What is my name, where do I live and what am I learning?')

SUMMARY:
 Priyansh from Jaipur is learning LangGraph. He asks for short facts: Paris—Eiffel Tower tallest building; Tokyo—most populous metro area. The assistant provides those facts. The conversation ends with a request to summarize, which we have now done. 



You: What is my name, where do I live and what am I learning?
AI:  Your name is **Priyansh**, you live in **Jaipur**, and you’re learning **LangGraph**.
     (stored 6 messages | summary: yes)

